# Notebook 07 — CLIP Zero-Shot Sky Classification
**Syllabus Week 12 | CLO 6: Vision-Language Models & Zero-Shot Learning**

## Problem Statement
Use OpenAI's **CLIP** (`openai/clip-vit-base-patch32`) to classify sky images
in a **zero-shot** manner — no training, no fine-tuning, no labelled images.

CLIP's vision encoder and text encoder are jointly trained to align image and
text embeddings. We exploit this by passing candidate labels as text prompts
and comparing their embeddings to the image embedding.

**Label mapping:** CLIP uses 6 fine-grained labels → mapped to 3 CNN labels
for compatibility with the `classify_sky_image` agent tool.

| CLIP label | CNN-compatible label |
|---|---|
| clear blue sky | sunny |
| bright sunshine | sunny |
| overcast cloudy sky | cloudy |
| sunset orange sky | cloudy |
| heavy rain clouds | rainy |
| stormy dark sky | rainy |

| Parameter | Value |
|---|---|
| Model | openai/clip-vit-base-patch32 |
| Vision backbone | ViT-B/32 |
| Embedding dim | 512 |
| Training required | None (zero-shot) |
| Input | URL or local image path |
| Output | CLIP label + CNN-compatible label + confidence |

## Section 2 — Math Derivation

### CLIP Contrastive Pre-training
CLIP is trained on (image, text) pairs with a symmetric cross-entropy loss.
For a batch of $N$ pairs, with image embeddings $I_i$ and text embeddings $T_j$:

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}\left[
  \log\frac{\exp(I_i \cdot T_i / \tau)}{\sum_j \exp(I_i \cdot T_j / \tau)}
  + \log\frac{\exp(T_i \cdot I_i / \tau)}{\sum_j \exp(T_j \cdot I_j / \tau)}
\right]$$

where $\tau$ is a learnable temperature parameter. This pushes matching pairs
close and non-matching pairs apart in embedding space.

### Zero-Shot Inference
Given a set of $C$ candidate label strings $\{l_1, \ldots, l_C\}$ and an image $v$:

1. Encode each label: $T_c = \text{TextEncoder}(\text{"a photo of a "} + l_c)$
2. Encode image: $I = \text{ImageEncoder}(v)$
3. L2-normalise both: $\hat{T}_c = T_c / \|T_c\|$, $\hat{I} = I / \|I\|$
4. Compute logits: $s_c = \hat{I} \cdot \hat{T}_c$
5. Softmax to probabilities: $p_c = \text{softmax}(s / \tau)$
6. Return $\arg\max_c p_c$ as the predicted label

### Why "a photo of a" prefix?
CLIP was trained on web image-caption pairs. Captions rarely start with a bare
noun — they are natural sentences. The prompt template aligns the zero-shot
label with the pre-training distribution, consistently improving accuracy.

### Label Mapping
After CLIP predicts one of its 6 fine-grained labels, `map_to_cnn_labels()`
collapses to 3 categories:
$$f(\text{clip\_label}) \in \{\text{sunny}, \text{cloudy}, \text{rainy}\}$$

The mapping is deterministic and table-driven — no additional inference needed.

In [ ]:
# Section 3 — Setup & Label Definitions
import sys; sys.path.insert(0, '..')

# CLIP labels and mapping
# from ml.infer.clip_infer import CLIP_LABELS, map_to_cnn_labels

# CLIP_LABELS = [
#     'clear blue sky',
#     'bright sunshine',
#     'overcast cloudy sky',
#     'sunset orange sky',
#     'heavy rain clouds',
#     'stormy dark sky',
# ]

# Verify mapping table:
# for label in CLIP_LABELS:
#     print(f'{label:25s} -> {map_to_cnn_labels(label)}')

# Expected output:
# clear blue sky            -> sunny
# bright sunshine           -> sunny
# overcast cloudy sky       -> cloudy
# sunset orange sky         -> cloudy
# heavy rain clouds         -> rainy
# stormy dark sky           -> rainy

In [ ]:
# Section 4 — Model Loading & Embedding Shapes
# TODO: run after pip install transformers Pillow

# from transformers import CLIPProcessor, CLIPModel
# import torch

# processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
# model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
# model.eval()
# n_params = sum(p.numel() for p in model.parameters())
# print(f'CLIP params: {n_params:,}')   # ~151M (vision + text encoders)

# Shape trace for one image and 6 labels:
# text_inputs   : {'input_ids': (6, 77), 'attention_mask': (6, 77)}
# image_inputs  : {'pixel_values': (1, 3, 224, 224)}
# text_embeds   : (6, 512)   L2-normalised
# image_embeds  : (1, 512)   L2-normalised
# logits        : (1, 6)     image @ text_embeds.T * exp(log_scale)
# probs         : (1, 6)     softmax

# Verify with a synthetic random image:
# from PIL import Image as PILImage
# import numpy as np
# fake_img = PILImage.fromarray(np.random.randint(0,255,(224,224,3),dtype=np.uint8))
# inputs = processor(text=CLIP_LABELS, images=fake_img, return_tensors='pt', padding=True)
# with torch.no_grad():
#     outputs = model(**inputs)
#     probs = outputs.logits_per_image.softmax(dim=1)
# print(f'probs shape: {probs.shape}')    # (1, 6)
# print(f'probs sum  : {probs.sum():.4f}')  # 1.0

In [ ]:
# Section 5 — Zero-Shot Inference on Sample Images
import asyncio

# from ml.infer.clip_infer import classify_sky_image

# Test with local synthetic images (generated during CNN training)
# test_images = [
#     '../datasets/sky_images/sunny/synthetic_0.jpg',
#     '../datasets/sky_images/cloudy/synthetic_0.jpg',
#     '../datasets/sky_images/rainy/synthetic_0.jpg',
# ]
# for img_path in test_images:
#     result = asyncio.run(classify_sky_image(img_path))
#     print(f'{img_path.split("/")[-2]:10s} -> '
#           f'CLIP: {result["clip_label"]} ({result["clip_confidence"]:.2%}) '
#           f'CNN-compat: {result["cnn_compatible_label"]}')

# TODO: paste results here
# sunny      -> CLIP: clear blue sky (XX.X%)  CNN-compat: sunny
# cloudy     -> CLIP: overcast cloudy sky (XX.X%)  CNN-compat: cloudy
# rainy      -> CLIP: heavy rain clouds (XX.X%)  CNN-compat: rainy

## Section 6 — Architecture Diagram

```
ZERO-SHOT INFERENCE (no training, no gradient updates)
──────────────────────────────────────────────────────

Image (URL or path)
      │
 CLIPProcessor.preprocess    224×224, normalise
      │  pixel_values: (1, 3, 224, 224)
 ViT-B/32 Vision Encoder
      │  image_embeds: (1, 512)   L2-normalised
      │
      │          6 candidate label strings
      │          "a photo of a <label>"
      │               │
      │     CLIPProcessor.tokenise  → (6, 77)
      │     Text Transformer Encoder
      │     text_embeds: (6, 512)  L2-normalised
      │
 logits = image_embeds @ text_embeds.T  * exp(log_scale)
      │  (1, 6)
 softmax → probabilities (1, 6)
      │
 argmax → CLIP label (one of 6)
      │
 map_to_cnn_labels()
      │
 CNN-compatible label: sunny | cloudy | rainy
      │
 {clip_label, clip_confidence, cnn_compatible_label}
```

In [ ]:
# Section 7 — Evaluation: Probability Bar Charts
import matplotlib.pyplot as plt
import numpy as np

# 7a. For each of the 3 synthetic test images, show a bar chart of
#     CLIP probability across all 6 labels

# from transformers import CLIPProcessor, CLIPModel
# from PIL import Image as PILImage
# import torch

# CLIP_LABELS = ['clear blue sky','bright sunshine','overcast cloudy sky',
#                'sunset orange sky','heavy rain clouds','stormy dark sky']

# model.eval()
# fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# for ax, (true_label, img_path) in zip(axes, [
#     ('sunny', test_images[0]),
#     ('cloudy', test_images[1]),
#     ('rainy', test_images[2])
# ]):
#     img = PILImage.open(img_path)
#     inputs = processor(text=CLIP_LABELS, images=img, return_tensors='pt', padding=True)
#     with torch.no_grad():
#         probs = model(**inputs).logits_per_image.softmax(dim=1)[0].numpy()
#     ax.barh(CLIP_LABELS, probs, color=['green' if p == probs.max() else 'steelblue' for p in probs])
#     ax.set_xlim(0, 1)
#     ax.set_xlabel('Probability')
#     ax.set_title(f'True: {true_label}')
# plt.suptitle('CLIP Zero-Shot Sky Classification')
# plt.tight_layout(); plt.show()

# 7b. Embedding similarity heatmap (6 labels × 6 labels)
# text_inputs = processor(text=CLIP_LABELS, return_tensors='pt', padding=True)
# with torch.no_grad():
#     text_embeds = model.get_text_features(**text_inputs)
#     text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)
# sim_matrix = (text_embeds @ text_embeds.T).numpy()
# plt.figure(figsize=(7, 6))
# plt.imshow(sim_matrix, cmap='coolwarm', vmin=-1, vmax=1)
# plt.colorbar(label='Cosine similarity')
# plt.xticks(range(6), [l.split()[-1] for l in CLIP_LABELS], rotation=45, ha='right')
# plt.yticks(range(6), [l.split()[-1] for l in CLIP_LABELS])
# plt.title('CLIP Text Embedding Similarity (label × label)')
# plt.tight_layout(); plt.show()

# TODO: paste accuracy on synthetic test images (expect ~100% since colours are distinctive)

## Section 8 — Comparison vs CNN & Reflection

### Results (fill in after running)
| Model | Training | Params | Test Accuracy | Latency |
|---|---|---|---|---|
| SkyCNN (NB04) | ~X min | < 2M | TODO % | TODO ms |
| **CLIP (ours)** | None (zero-shot) | ~151M | TODO % | TODO ms |

### Design Decisions
| Decision | Alternative | Reason |
|---|---|---|
| 6 fine-grained CLIP labels | 3 labels directly | More discriminative prompts; 6→3 mapping post-hoc |
| "a photo of a" prompt prefix | Bare label string | Aligns with CLIP pre-training distribution |
| sunset → cloudy mapping | sunset → sunny | Sunset produces warm-toned overcast; ambiguous but cloudy is safer |
| asyncio.to_thread | Sync call | CLIP inference blocks the event loop; wrapping keeps FastAPI responsive |
| Lazy model load | Preload at startup | CLIP is 600MB — skip loading if tool never called |

### Zero-Shot vs Fine-Tuned Trade-offs
| Aspect | Zero-Shot CLIP | Fine-Tuned CNN |
|---|---|---|
| Training data | None | Labelled images required |
| Generalisation | High (ImageNet-21k + LAION) | Limited to training distribution |
| Model size | ~151M params, ~600MB | < 2M params, < 8MB |
| CPU latency | ~500–2000ms | < 50ms |
| Label flexibility | Change labels without retraining | Retrain required for new classes |
| Interpretability | Attention maps via ViT | Conv filter visualisations |

### Why CLIP + CNN Both?
- **CLIP** handles arbitrary real-world sky photos — no labelled dataset needed.
- **CNN** would be faster and smaller once trained on real data.
- **Hybrid:** CLIP labels feed into the CNN-compatible taxonomy, so both outputs
  are interchangeable in the agent tool interface.

### Limitations
- CLIP was not trained on Pakistani sky images specifically — regional haze
  and smog patterns may confuse the model.
- 6-label taxonomy cannot distinguish drizzle from heavy monsoon rain.
- No confidence calibration — softmax probabilities from CLIP are overconfident.

### Numbers to fill in
- CLIP accuracy on synthetic test images: **TODO** (expect high — synthetic colours are extreme)
- Average inference latency: **TODO ms**
- Most confused label pair: **TODO** (expect cloudy/rainy)